In [ ]:
!pip install xgboost catboost faiss-cpu

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

os.listdir('/content/drive/MyDrive')

In [ ]:
import os

os.listdir('/content/drive/MyDrive/ML_Datasets')

In [ ]:
import pandas as pd

data = pd.read_excel(r'/content/drive/MyDrive/ML_Datasets/South Africa Road Accidents Dataset - 2017.xlsx')

data.head()

In [ ]:
import pandas as pd

data = pd.read_excel(r'/content/drive/MyDrive/ML_Datasets/South Africa Road Accidents Dataset - 2017.xlsx')

print("dataset shape:")
print(data.shape)

#show column names
print("\nColumns Names:")
print(data.columns)

#show missing vaues
print("\nMissing values:")
print(data.isnull().sum())

In [ ]:
print(data['Accident Severity'].value_counts())


In [ ]:
#inpud data which is a feature
X=data.drop('Accident Severity',axis=1)

#target which what we want to predict
y= data['Accident Severity']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
#creating an encoder
label_encoder=LabelEncoder()

# encode all categorical columns
for column in X.columns:
  if X[column].dtype == 'object':
    # Convert problematic datetime objects to string before encoding
    X[column] = X[column].astype(str)
    X[column] = label_encoder.fit_transform(X[column])

# Encode target variables once
y=label_encoder.fit_transform(y)

#view encoded data
X.head()

In [ ]:
#convert datw into datetime format
X['Date']=pd.to_datetime(X['Date'])

#extracting usefull data features
X['Year']=X['Date'].dt.year
X['Month']=X['Date'].dt.month
X['Day']=X['Date'].dt.day

#drop the original date column
X=X.drop('Date',axis=1)

#view the data
X.head()

In [ ]:
from sklearn.model_selection import train_test_split
#split dataset
X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#check shapes
print("Training Features shape", X_train.shape)
print("Training Labels shape", y_train.shape)
print("Testing Features shape", X_test.shape)
print("Testing Labels shape", y_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#create a model
RandomForestClassifier_model=RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

#Train model
RandomForestClassifier_model.fit(X_train,y_train)

#predictions made
y_pred_rfdm=RandomForestClassifier_model.predict(X_test)

print(y_pred_rfdm)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
    )
#Accuracy
accuracy=accuracy_score(y_test,y_pred_rfdm)
print("Accuracy:",accuracy)

#Precision
precision=precision_score(y_test,y_pred_rfdm,average='weighted')
print("Precision:",precision)

#Recall
recall=recall_score(y_test,y_pred_rfdm,average='weighted')
print("Recall:",recall)

#f1
f1=f1_score(y_test,y_pred_rfdm,average='weighted')
print("F1 Score:",f1)

#results
print("Accuracy",accuracy)
print("Precision",precision)
print("Recall",recall)
print("F1 Score",f1)

#classification report
print("\nClassification Report:")
print(classification_report(y_test,y_pred_rfdm))

#confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test,y_pred_rfdm))





In [ ]:
from xgboost import XGBClassifier
#creating a XGBoost model
xgb_model=XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='mlogloss'
 )

#train model
xgb_model.fit(X_train,y_train)

#predictions made
y_pred_xgb=xgb_model.predict(X_test)

print(y_pred_xgb)

In [ ]:
#Accuracy
xgb_accuracy=accuracy_score(y_test,y_pred_xgb)
print("Accuracy:",xgb_accuracy)

#Precision
xgb_precision=precision_score(y_test,y_pred_xgb,average='weighted')
print("Precision:",xgb_precision)

#Recall
xgb_recall=recall_score(y_test,y_pred_xgb,average='weighted')
print("Recall:",xgb_recall)

#f1
xgb_f1=f1_score(y_test,y_pred_xgb,average='weighted')
print("F1 Score:",xgb_f1)



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix for the Random Forest model
cm_rf = confusion_matrix(y_test, y_pred_rfdm)

#plotting confusion matrix
plt.figure(figsize=(8,6))
sns.heatmap(
cm_rf,
annot=True,
fmt='d',
cmap='Blues'
)

plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

In [ ]:
#3. Reinforcement Learning for Accident Prevention
import numpy as np
#states
states=["Lower Risk", "Medium Risk", "High Risk"]

#actions
actions=["Do Nothing", "Safety Campaign", "Increase Enforcement"]

#reward matrix
reward_matrix=np.array([
    [5, 8, 3],
    [2, 10, 7],
    [1, 4, 15]
])
print("Reward Matrix: ")
print(reward_matrix)


In [ ]:
#Q-table
q_table=np.zeros((len(states),len(actions)))
print("Q-table: ")
print(q_table)

In [ ]:
#parameters

#Learning rate control
alpha=0.1
#Discount factor control
gemma=0.9
# episodes (one complete learning attempt)
episodes=1000



#Training
for  episode in range(episodes):
  #choose random state
  state=np.random.randint(0, len(states))

  #choose random action
  action=np.random.randint(0, len(actions))

  #get reward
  reward=reward_matrix[state, action]

  #Update Q-value
  q_table[state, action]=q_table[state, action]+ alpha * (
      reward + gemma * np.max(q_table[state])-q_table[state, action]
  )
print("Trained Q-Table: ")
print(q_table)



"""convergence means Q-Values stop changing eventually
Convergence occurs when the agent learn a stable policy and when further training produces very small changes.
"""

In [ ]:
#find best action for each state
best_actions=np.argmax(q_table, axis=1)

for state_idx, action_idx in enumerate(best_actions):
  print(
      f"state: {states[state_idx]} ->Best Action: {actions[action_idx]}"
  )


In [ ]:
#System Integration
risk_mapping={
    0: "Medium Risk",
    1:"Lower Risk",
    2:"High Risk",
}

policy={
    "Lower Risk": "Safety Compaign",
    "Medium Risk": "Safety Campaign",
    "High Risk": "Increase Enforcement"
}

for prediction in y_pred_rfdm[: 10]:
  risk_state=risk_mapping[prediction]

  action=policy[risk_state]

  print(
      f"Predicted State: {risk_state} -> Recommended Action: {action} "

  )


In [ ]:
!pip install pymupdf

In [ ]:
import fitz
import pandas as pd
import os

In [ ]:
import fitz
import os

# Define the directory where PDF files are located
directory_path = "/content/drive/MyDrive/ML_Datasets"

pdf_files = []
for file_name in os.listdir(directory_path):
  if file_name.endswith('.pdf'):
    pdf_files.append(os.path.join(directory_path, file_name))

all_text=""

for pdf_file in pdf_files:
  # Check if the file exists and is not empty
  if os.path.exists(pdf_file) and os.path.getsize(pdf_file) > 0:
    try:
      doc = fitz.open(pdf_file)
      for page in doc:
        all_text += page.get_text()
      doc.close()
    except ValueError as e:
      print(f"Skipping encrypted or corrupted file: {pdf_file} - Error: {e}")
    except Exception as e:
      print(f"Skipping {pdf_file} due to an unexpected error: {e}")
  else:
    print(f"Skipping empty or non-existent file: {pdf_file}")

print(" Total number of texts extracted: ", len(all_text))
print("\nFirst 1000 characters of the extracted text:")
print(all_text[:1000])

In [ ]:
import re
#covert to lowercase
clean_text=all_text.lower()

#remove page references
clean_text=re.sub(r'page:\s\d+', ' ',clean_text)

#remove extra whitespace
clean_text=re.sub(r'\s+', ' ',clean_text)

#remove all special characters
clean_text=re.sub(r'[^a-zA-Z0-9\s]', ' ',clean_text)

#remove numbers
clean_text=re.sub(r'\d+', ' ',clean_text)

#removing extra space again
clean_text=re.sub(r'\s+', ' ',clean_text).strip()

print("Cleaned text length: ", len(clean_text))
print("\nFirst 1000 characters of the cleaned text:")
print(clean_text[:1000])

In [ ]:
!pip install transformers sentence-transformers

In [ ]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)
tokens=tokenizer.tokenize(
    clean_text[:1000]
)
print("Number of tokens: ", len(tokens))
print("first 50 tokens: \n")
print(tokens[:50])

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model=SentenceTransformer(
    'all-MiniLM-L6-v2'
)
#split text into chunks
chunk_size=500

text_chunks=[
    clean_text[i:i+chunk_size]
    for i in range(0, len(clean_text), chunk_size)
]

print("Number of text chunks: ", len(text_chunks))

#embedding for first 100 chunks
sample_chunks=text_chunks[:100]

embeddings=embedding_model.encode(
    sample_chunks,
    show_progress_bar=True
)

print("Embedding Shape: ", embeddings.shape)


In [ ]:
#view chunks transcript
for i in range(10):
  print(f"\n--Chunks{i}---\n")
  print(text_chunks[i][:500])

In [ ]:
import pandas as pd

sentiment_data = []

for chunk in text_chunks[:500]:

    text = chunk[:500].lower()

    if any(word in text for word in [
        "failed",
        "corruption",
        "crisis",
        "poverty",
        "unemployment"
    ]):
        sentiment_data.append([chunk[:500], "negative"])

    elif any(word in text for word in [
        "improve",
        "improved",
        "achievement",
        "successful",
        "progress",
        "support"
    ]):
        sentiment_data.append([chunk[:500], "positive"])

    elif any(word in text for word in [
        "committee",
        "meeting",
        "assembly",
        "chairperson",
        "report"
    ]):
        sentiment_data.append([chunk[:500], "neutral"])

sentiment_df = pd.DataFrame(
    sentiment_data,
    columns=["text", "sentiment"]
)

print(sentiment_df.shape)

print("\nClass Distribution:\n")
print(sentiment_df["sentiment"].value_counts())

In [ ]:
sentiment_df["sentiment"].value_counts()

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder=LabelEncoder()
sentiment_df['label']=label_encoder.fit_transform(
    sentiment_df['sentiment']
)

print(label_encoder.classes_)
print("\n")
print(sentiment_df.head())

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    sentiment_df['text'].tolist(),
    sentiment_df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=sentiment_df['label']
)

print("Training samples:", len(train_texts))
print("Testing samples:", len(test_texts))

In [ ]:
print(sentiment_df.shape)


In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    'distilbert-base-uncased'
)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenisation complete")

In [ ]:
import torch

class SentimentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = SentimentDataset(
    train_encodings,
    train_labels
)

test_dataset = SentimentDataset(
    test_encodings,
    test_labels
)

print("Training dataset size:", len(train_dataset))
print("Testing dataset size:", len(test_dataset))

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

print("Model loaded successfully")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=10
)

print("Training arguments created")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("Trainer created successfully")

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)

predicted_labels = predictions.predictions.argmax(axis=1)

print(predicted_labels[:10])

In [ ]:
print("Actual Labels:")
print(test_labels[:20])

print("\nPredicted Labels:")
print(predicted_labels[:20])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(
    test_labels,
    predicted_labels
)

precision = precision_score(
    test_labels,
    predicted_labels,
    average='weighted',
    zero_division=0
)

recall = recall_score(
    test_labels,
    predicted_labels,
    average='weighted',
    zero_division=0
)

f1 = f1_score(
    test_labels,
    predicted_labels,
    average='weighted',
    zero_division=0
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(
    classification_report(
        test_labels,
        predicted_labels,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    test_labels,
    predicted_labels
)

print(cm)

In [ ]:
import faiss
import numpy as np

# Convert embeddings
embeddings = np.array(embeddings).astype('float32')

# index
index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

# Store vectors
index.add(embeddings)

print("Vectors stored:", index.ntotal)

In [ ]:
query = "What was said about education policy?"

query_embedding = embedding_model.encode(
    [query]
).astype('float32')

k = 5

distances, indices = index.search(
    query_embedding,
    k
)

print("Retrieved Chunks:\n")

for idx in indices[0]:
    print("\n-------------------\n")
    print(text_chunks[idx][:1000])

In [ ]:
# first 500 chunks

sample_chunks = text_chunks[:500]

embeddings = embedding_model.encode(
    sample_chunks,
    show_progress_bar=True
)

print(embeddings.shape)

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(embeddings)

print("Vectors stored:", index.ntotal)

In [ ]:
query = "What was said about education policy?"

query_embedding = embedding_model.encode(
    [query]
).astype("float32")

distances, indices = index.search(
    query_embedding,
    5
)

for i, idx in enumerate(indices[0]):

    print(f"\n=== Result {i+1} ===\n")

    print(sample_chunks[idx][:1000])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_gen = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_gen = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("Tokenizer and Model loaded for generative QA")

In [ ]:
# Cell functionality moved to direct model/tokenizer calls in cell `kuFIGNALmxW5`.
# This cell is no longer needed.

In [ ]:
retrieved_context = "\n\n".join(
    [sample_chunks[idx] for idx in indices[0]]
)

prompt = f"""
Answer the question using only the provided context.

Context:
{retrieved_context}

Question:
What was said about education policy?

Answer:
"""

print(prompt[:1500])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_gen = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

model_gen = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("FLAN-T5 loaded successfully")

In [ ]:
input_ids = tokenizer_gen(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).input_ids

outputs = model_gen.generate(
    input_ids,
    max_new_tokens=100
)

answer = tokenizer_gen.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

In [ ]:
print(sentiment_df)

In [ ]:
sentiment_df.to_csv(
    "hansard_sentiment_dataset.csv",
    index=False
)

print("Dataset saved successfully")

In [ ]:
from google.colab import files

files.download("hansard_sentiment_dataset.csv")

In [ ]:
import nbformat

notebook_path = "Machine_Learning700_Assignment.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open("fixed_notebook.ipynb", "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Notebook fixed")